# Phase 7 — Metric Scale Analysis

## What VGGT's scale actually is

VGGT does **not** output absolute metric scale.  During training, each scene is
normalised so the average 3-D point depth equals 1.0 (scene-unit normalisation).
The model learns this canonical normalisation from data — so at inference it
outputs poses in *normalised scene units*, not metres.

All previous ATE runs used `with_scale=True`, which silently absorbs this
discrepancy via a Umeyama scale factor `s`.  That `s` is the key quantity:

| s value | Interpretation |
|---------|----------------|
| s ≈ 1.0 | VGGT scale matches GT metric scale — lucky coincidence |
| s > 1.0 | VGGT under-scales the scene (predictions too small) |
| s < 1.0 | VGGT over-scales the scene (predictions too large) |
| s stable across runs | Consistent normalisation — can be calibrated out |
| s varies across runs | Unreliable scale — cannot be corrected post-hoc |

## Experiments in this notebook

1. **Scale factor vs resolution** — is `s` stable or resolution-dependent?
2. **Scaled vs unscaled ATE** — how much of ATE is pure scale error?
3. **Scale factor across sequences** — is `s` scene-dependent?
4. **Paper-faithful AUC** — pairwise relative pose AUC@(5°,10°,20°,30°),
   scale-invariant by construction, matching the VGGT paper evaluation protocol.

## 0 — Setup

In [ ]:
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

def git(*args):
    subprocess.check_call(["git"] + list(args))

pip("--upgrade", "numpy")
pip("plotly", "pandas", "pillow", "scipy", "matplotlib")

if not os.path.isdir("vggt"):
    git("clone", "--depth", "1", "https://github.com/facebookresearch/vggt.git")
    pip("-e", "vggt")

if not os.path.isdir("vggt-eval"):
    git("clone", "--depth", "1", "https://github.com/insha-py/vggt-eval.git")
else:
    git("-C", "vggt-eval", "pull", "--ff-only")

sys.path.insert(0, "vggt-eval")
sys.path.insert(0, "vggt")

print("Packages installed. Restarting kernel ...")
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
except Exception:
    print("Restart the kernel manually, then run from the Imports cell.")

## 1 — Imports

In [ ]:
import os, sys, gc
for p in ["vggt-eval", "vggt"]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.tum_vi            import TUMVIDataset
from src.resolution_sweep  import ResolutionSweeper, DEFAULT_RESOLUTIONS
from src.metrics           import (
    compute_ate, compute_rpe,
    compute_relative_pose_auc,
    _camera_centers,
)

np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2 — Download Data & Load Model

In [ ]:
SEQUENCE     = "room1"
N_FRAMES     = 8
DOWNLOAD_DIR = "/tmp/tumvi"

ds   = TUMVIDataset(sequence=SEQUENCE, n_frames=N_FRAMES, download_dir=DOWNLOAD_DIR)
ds.download()
data = ds.load()

image_dir     = data["image_dir"]
gt_extrinsics = data["gt_extrinsics"]   # (N, 3, 4) camera frame, metric scale
N = len(gt_extrinsics)
print(f"Loaded {N} frames   GT shape: {gt_extrinsics.shape}")

In [ ]:
# Single-pass, store extrinsics so we can run metrics multiple ways
sweeper = ResolutionSweeper(
    resolutions      = DEFAULT_RESOLUTIONS,
    conf_thresh      = 5.0,
    store_extrinsics = True,   # keep raw poses for custom metric calculations
)
sweeper.load_model()

## 3 — Resolution Sweep with Scale Factor Logging

In [ ]:
# with_scale=True → Umeyama estimates s; scale_factor captured in each ResolutionResult
results = sweeper.run(
    image_dir     = image_dir,
    max_frames    = N_FRAMES,
    gt_extrinsics = gt_extrinsics,
    align         = True,
    with_scale    = True,
)
df = sweeper.to_dataframe(results)

print("\nSweep results (with scale factors):")
print(df[["resolution", "ate_mean", "ate_rmse", "scale_factor",
          "rpe_trans", "rot_mean_deg"]].to_string(
    index=False, float_format="{:.4f}".format))

## 4 — Scale Factor vs Resolution

If `s` is stable across resolutions, VGGT uses a consistent normalisation
regardless of input size — the scale can be calibrated out with a single
correction factor.  If `s` varies, scale is resolution-dependent and cannot
be corrected post-hoc without knowing the resolution.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["scale_factor"],
    mode="lines+markers",
    line=dict(color="royalblue", width=2), marker=dict(size=10),
    name="Scale factor s",
))
fig.add_hline(y=1.0, line_dash="dash", line_color="green",
              annotation_text="s=1 (perfect metric scale)",
              annotation_position="bottom right")
fig.add_vline(x=518, line_dash="dash", line_color="grey",
              annotation_text="Native 518 px", annotation_position="top left")
fig.update_layout(
    title=f"Umeyama Scale Factor s vs Resolution — TUM-VI {SEQUENCE} ({N} frames)",
    xaxis_title="Resolution (px)",
    yaxis_title="Scale factor s  (VGGT units → GT metric units)",
)
fig.show()

s_mean = df["scale_factor"].mean()
s_std  = df["scale_factor"].std()
s_cv   = s_std / s_mean * 100  # coefficient of variation
print(f"Scale factor:  mean={s_mean:.4f}  std={s_std:.4f}  CV={s_cv:.2f}%")
print(f"Deviation from metric (s=1):  {abs(s_mean - 1.0)*100:.1f}%")
if s_cv < 5:
    print("→ Scale is STABLE across resolutions (CV < 5%) — a single calibration factor suffices.")
else:
    print("→ Scale is VARIABLE across resolutions — resolution-dependent calibration needed.")

## 5 — Scaled vs Unscaled ATE

**Scaled ATE** (`with_scale=True`): Umeyama finds the best `s`, hiding scale error.  
**Unscaled ATE** (`with_scale=False`): `s` is fixed at 1.0 — scale error contributes fully.

The gap between the two lines is the ATE cost of not correcting for scale.

In [ ]:
# Run unscaled version (align=True but with_scale=False)
results_noscale = sweeper.run(
    image_dir     = image_dir,
    max_frames    = N_FRAMES,
    gt_extrinsics = gt_extrinsics,
    align         = True,
    with_scale    = False,
)
df_noscale = sweeper.to_dataframe(results_noscale)

# Also run completely unaligned
results_noalign = sweeper.run(
    image_dir     = image_dir,
    max_frames    = N_FRAMES,
    gt_extrinsics = gt_extrinsics,
    align         = False,
    with_scale    = False,
)
df_noalign = sweeper.to_dataframe(results_noalign)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["ate_mean"],
    mode="lines+markers", name="ATE (align+scale)",
    line=dict(color="green", width=2), marker=dict(size=8),
))
fig.add_trace(go.Scatter(
    x=df_noscale["resolution"], y=df_noscale["ate_mean"],
    mode="lines+markers", name="ATE (align, no scale)",
    line=dict(color="royalblue", width=2, dash="dash"), marker=dict(size=8),
))
fig.add_trace(go.Scatter(
    x=df_noalign["resolution"], y=df_noalign["ate_mean"],
    mode="lines+markers", name="ATE (no align, no scale)",
    line=dict(color="tomato", width=2, dash="dot"), marker=dict(size=8),
))
fig.add_vline(x=518, line_dash="dash", line_color="grey",
              annotation_text="518 px", annotation_position="top left")
fig.update_layout(
    title=f"Scaled vs Unscaled ATE — TUM-VI {SEQUENCE} ({N} frames)",
    xaxis_title="Resolution (px)",
    yaxis_title="ATE mean (m)",
    legend=dict(x=0.01, y=0.99),
)
fig.show()

print("\nATE comparison at 518 px (native resolution):")
r518 = df.loc[df["resolution"] == 518]
r518_ns = df_noscale.loc[df_noscale["resolution"] == 518]
r518_na = df_noalign.loc[df_noalign["resolution"] == 518]
print(f"  align+scale  : {r518['ate_mean'].values[0]:.4f} m")
print(f"  align, no s  : {r518_ns['ate_mean'].values[0]:.4f} m")
print(f"  no align     : {r518_na['ate_mean'].values[0]:.4f} m")

scale_contribution = (r518_ns['ate_mean'].values[0] - r518['ate_mean'].values[0])
print(f"  Scale error contribution to ATE : {scale_contribution:.4f} m  "
      f"({100*scale_contribution/r518_ns['ate_mean'].values[0]:.1f}% of no-scale ATE)")

## 6 — Scale Factor Across Sequences

Is `s` scene-dependent?  A consistent `s` across scene types means VGGT's
normalisation is dominated by the training-data prior, not the specific scene.
A varying `s` means per-scene calibration is required.

In [ ]:
SEQUENCES    = ["room1", "room2", "corridor1", "slides1"]
scale_rows   = []
os.makedirs("results", exist_ok=True)

for seq in SEQUENCES:
    print(f"\n--- {seq} ---")
    ds_s = TUMVIDataset(sequence=seq, n_frames=N_FRAMES, download_dir=DOWNLOAD_DIR)
    ds_s.download()
    data_s = ds_s.load()

    res_s = sweeper.run(
        image_dir     = data_s["image_dir"],
        max_frames    = N_FRAMES,
        gt_extrinsics = data_s["gt_extrinsics"],
        align         = True,
        with_scale    = True,
    )
    df_s = sweeper.to_dataframe(res_s)
    df_s["sequence"] = seq

    # Summarise scale factor: mean and std across resolutions
    s_mean = df_s["scale_factor"].mean()
    s_std  = df_s["scale_factor"].std()
    print(f"  scale_factor: mean={s_mean:.4f}  std={s_std:.4f}")

    for _, row in df_s.iterrows():
        scale_rows.append({
            "sequence":     seq,
            "resolution":   int(row["resolution"]),
            "scale_factor": row["scale_factor"],
            "ate_scaled":   row["ate_mean"],
        })

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_scale = pd.DataFrame(scale_rows)
df_scale.to_csv("results/phase7_scale_across_sequences.csv", index=False)
print("\nSaved: results/phase7_scale_across_sequences.csv")

In [ ]:
colors = px.colors.qualitative.Plotly

fig = go.Figure()
for i, seq in enumerate(SEQUENCES):
    sub = df_scale[df_scale["sequence"] == seq]
    fig.add_trace(go.Scatter(
        x=sub["resolution"], y=sub["scale_factor"],
        mode="lines+markers", name=seq,
        line=dict(color=colors[i], width=2), marker=dict(size=8),
    ))

fig.add_hline(y=1.0, line_dash="dash", line_color="green",
              annotation_text="s=1 (perfect metric)",
              annotation_position="bottom right")
fig.update_layout(
    title="Scale Factor s per Sequence vs Resolution (s=1 means metric scale recovered)",
    xaxis_title="Resolution (px)",
    yaxis_title="Umeyama scale factor s",
    legend=dict(x=0.01, y=0.99),
)
fig.show()

# Summary: per-sequence mean scale and deviation from 1.0
summary = (df_scale.groupby("sequence")["scale_factor"]
           .agg(["mean", "std"])
           .rename(columns={"mean": "s_mean", "std": "s_std"}))
summary["s_error_pct"] = abs(summary["s_mean"] - 1.0) * 100
summary["s_cv_pct"]    = summary["s_std"] / summary["s_mean"] * 100
print("\nPer-sequence scale summary:")
print(summary.to_string(float_format="{:.4f}".format))

## 7 — Paper-Faithful AUC: Pairwise Relative Pose

The VGGT paper reports **AUC@30** combining Relative Rotation Accuracy (RRA)
and Relative Translation Accuracy (RTA) over all frame pairs.  Unlike ATE,
this metric is **scale-invariant by construction** — it measures angular errors
in rotation and translation direction, not metric distance.  This is the right
metric for comparing to the paper's reported numbers.

In [ ]:
THRESHOLDS = [5, 10, 20, 30]
auc_rows   = []

for rr in results:   # results from the initial room1 sweep
    if rr.extrinsics is None:
        continue
    gt = gt_extrinsics[:len(rr.extrinsics)]
    auc = compute_relative_pose_auc(rr.extrinsics, gt, thresholds_deg=THRESHOLDS)
    row = {"resolution": rr.resolution}
    row.update(auc)
    auc_rows.append(row)

df_auc = pd.DataFrame(auc_rows)
print("\nPairwise Relative Pose AUC — TUM-VI room1:")
print(df_auc.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
auc_cols = [c for c in df_auc.columns if c.startswith("auc@")]
acc_cols = [c for c in df_auc.columns if c.startswith("acc@")]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("AUC@θ vs Resolution", "Acc@θ vs Resolution"))

palette = ["#e41a1c", "#ff7f00", "#4daf4a", "#377eb8"]
for col_name, color in zip(auc_cols, palette):
    fig.add_trace(go.Scatter(
        x=df_auc["resolution"], y=df_auc[col_name],
        mode="lines+markers", name=col_name,
        line=dict(color=color, width=2), marker=dict(size=8)),
        row=1, col=1)

for col_name, color in zip(acc_cols, palette):
    fig.add_trace(go.Scatter(
        x=df_auc["resolution"], y=df_auc[col_name],
        mode="lines+markers", name=col_name,
        line=dict(color=color, width=2, dash="dash"), marker=dict(size=8)),
        row=1, col=2)

fig.update_xaxes(title_text="Resolution (px)")
fig.update_yaxes(title_text="AUC (normalised)", row=1, col=1)
fig.update_yaxes(title_text="Fraction of pairs", row=1, col=2)
fig.update_layout(
    height=430,
    title_text="Pairwise Relative Pose AUC — paper protocol (scale-invariant)",
)
fig.show()

## 8 — Summary Table & Save

In [ ]:
# Merge all results into one table
df_full = df[["resolution", "ate_mean", "ate_rmse", "scale_factor",
              "rpe_trans", "rot_mean_deg"]].merge(
    df_noscale[["resolution", "ate_mean"]].rename(
        columns={"ate_mean": "ate_noscale"}),
    on="resolution"
).merge(
    df_auc[["resolution"] + auc_cols + acc_cols],
    on="resolution"
)

df_full["scale_error_pct"] = abs(df_full["scale_factor"] - 1.0) * 100
df_full["ate_scale_contrib"] = df_full["ate_noscale"] - df_full["ate_mean"]

print(f"\n=== Metric Scale Summary — TUM-VI {SEQUENCE} ({N} frames) ===")
print(df_full.to_string(index=False, float_format="{:.4f}".format))

df_full.to_csv(f"results/phase7_metric_scale_{SEQUENCE}.csv", index=False)
print(f"\nSaved to results/phase7_metric_scale_{SEQUENCE}.csv")

## 9 — Interpretation Guide

Read the results using the following framework:

### Scale factor `s`
- **Stable across resolutions** (CV < 5%): resolution does not affect VGGT's scene normalisation. A single global correction factor can convert VGGT outputs to metric scale.
- **Varies across sequences**: VGGT's scale depends on scene content (average depth). This is expected — scenes with distant objects get a different normalisation than close-up scenes.
- **s ≈ 1.0**: the training-data normalisation happens to match TUM-VI's metric scale. Unlikely; more probable that s >> 1 or s << 1.

### Scaled vs unscaled ATE gap
- Large gap → most of the trajectory error in `with_scale=True` runs is actually scale error, not pose error.
- Small gap → VGGT's scale is already close to GT and pose error dominates.

### AUC (pairwise, scale-invariant)
- This is the fair comparison to the VGGT paper's reported numbers.
- Resolution invariance should hold here too — if AUC is flat, VGGT's relative pose quality does not depend on resolution.
- Higher AUC@30 is better (1.0 = all pairs within 30°).